# CaP-X VLA worker — Colab / GPU Jupyter

共通VLA workerのGPU側。最初の実モデルはOpenVLA。LIBEROシミュレータとCaP-XのPython実行器は手元で動かします。

1. repoの `scripts/package_vla_worker.py` で作った `vla-worker.zip` をGPU側に置く。
2. 手元で `python -m capx.extensions.vla.demo --output output/vla-live --public-url <GPUから到達できるURL>` を起動する。
3. 生成された `connection.json` をGPU側に置く。tokenが入るのでセル出力やGitへ保存しない。

共有Colab L4 pool所有のruntimeでは、このNotebookを直接実行せずqueueを使う。
通常のColabまたは自分のLinux CUDA Jupyter向け。モデル重みのdownloadと専用venvの作成を行う。


In [ ]:
from pathlib import Path
import json, sys, zipfile

# Colabでは左のFilesから2ファイルをアップロード。Jupyterでは下のpathを変更する。
source_zip = Path("vla-worker.zip")
connection_file = Path("connection.json")
root = Path("capx-vla-worker").resolve()
root.mkdir(exist_ok=False)
with zipfile.ZipFile(source_zip) as archive:
    for name in archive.namelist():
        if not (root / name).resolve().is_relative_to(root):
            raise ValueError("Invalid source archive path")
    archive.extractall(root)
sys.path.insert(0, str(root))
connection = json.loads(connection_file.read_text())


## モデルを選ぶ

`adapter="openvla"` は組込みadapter。外部モデルは `my_package.my_adapter:MyVlaAdapter` の形で指定し、
package・checkpoint・モデル固有の依存ファイルとaction変換を用意する。
外部adapterは `requirements` と別の `environment_name` も指定する。詳細は `docs/vla-osc.md`。

OpenVLAの初回downloadには時間がかかる。`policy_ready` が表示されてから手元の実行が進む。
Notebook本体にtorchをinstallせず、専用Python 3.10 processで実行する。


In [ ]:
from capx.extensions.vla.gpu_setup import run_worker

adapter = "openvla"
adapter_config = None
requirements = None  # 外部adapterでは依存ファイルの絶対pathを指定
environment_name = "openvla-policy"

run_worker(connection, root=root, output=root / "output", seconds=600,
           adapter=adapter, adapter_config=adapter_config,
           requirements=requirements, environment_name=environment_name)


In [ ]:
# tokenや要求画像を表示せず、実際の推論件数だけ確認する。
result = json.loads((root / "output/worker/result.json").read_text())
print(result)


手元にはbefore/after画像、actionログ、有限デモの映像が保存されます。
GPUへの要求数と環境へ実際に適用したstep数は別なので、手元の `report.json` も確認します。

cell完了後はconnectionファイルを取り除き、Colabでは「接続解除してランタイムを削除」、
GPU Jupyterでは自分が確保したserver/runtimeを終了してください。他の人や別作業の割当は止めません。


In [ ]:
connection_file.unlink(missing_ok=True)
connection = None
